# Step-Level NLI — Ground Truth × Gherkin Generations (JSON)

This notebook calculates **only step-level NLI for Gherkin scenarios**. No global scenario-level NLI is calculated.

The objective is to produce, for each generated scenario in each execution, a single `nli_score` representing its **semantic proximity to the reference scenario**, while also preserving completeness and contradiction indicators.

## Strategy

1. Extract `Given`, `When`, `Then`, `And`, and `But` from the ground truth and the generation.
2. `And`/`But` inherit the previous semantic context (`Given`, `When`, or `Then`).
3. Align steps by **semantic group + position within the group**.
4. For each aligned pair, calculate NLI in both directions: reference → generation and generation → reference.
5. For each direction:

\[
S(A,B)=P(E)+0.5P(N)
\]

Since \(P(E)+P(N)+P(C)=1\), equivalently:

\[
S(A,B)=\frac{1+P(E)-P(C)}{2}
\]

6. The bidirectional score for step \(i\) is:

\[
NLI_i=\frac{S(R_i,G_i)+S(G_i,R_i)}{2}
\]

7. The final scenario score is:

\[
NLI_{scenario}=\frac{1}{m}\sum_{i=1}^{m}NLI_i
\]

where \(m\) is the number of aligned steps.

**The higher the `nli_score`, the greater the semantic proximity.**

## Coverage

Coverage is reported separately:

\[
coverage=\frac{aligned\ steps}{ground\ truth\ steps}
\]

Extra steps do not increase coverage and are counted separately.

## Output

The notebook generates **a single CSV**, with one row per scenario/execution. No global NLI is calculated or exported.


In [ ]:
# ============================================================
# 1. DEPENDENCY INSTALLATION
# ============================================================

!pip -q install -U transformers sentencepiece accelerate safetensors

In [ ]:
# ============================================================
# 2. IMPORTS AND CONFIGURATION
# ============================================================

import json
import re
import os
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# NLI is applied to steps, not to the entire scenario.
MAX_LENGTH = 192
NLI_DECIMAL_PLACES = 6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE_GPU = 64
BATCH_SIZE_CPU = 16
BATCH_SIZE = BATCH_SIZE_GPU if DEVICE.type == "cuda" else BATCH_SIZE_CPU

USE_MIXED_PRECISION = True
AUTO_DOWNLOAD_CSV = True

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✓ GPU detected: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    print("⚠ WARNING: no GPU was detected.")
    print("  mDeBERTa-v3-base with bidirectional NLI on CPU may take hours.")
    print("  In Google Colab: Runtime > Change runtime type > T4 GPU.")
    torch.set_num_threads(min(8, os.cpu_count() or 1))

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"NLI model: {MODEL_NAME}")

In [ ]:
# ============================================================
# 3. UPLOAD AND AUTOMATIC JSON IDENTIFICATION
# ============================================================

def load_json_bytes(file_name, content):
    try:
        text = content.decode("utf-8-sig")
        return json.loads(text)
    except Exception as e:
        raise ValueError(f"Could not read '{file_name}' as JSON: {e}") from e


def classify_json(file_name, data):
    if not isinstance(data, dict):
        return "desconhecido"

    cases_data = data.get("cases")
    if not isinstance(cases_data, list):
        return "desconhecido"

    if data.get("is_reference_base") is True:
        return "ground_truth"

    if cases_data:
        first_item = cases_data[0]
        if isinstance(first_item, dict):
            if "generations" in first_item:
                return "geracoes"
            if "reference_id" in first_item and "gherkin" in first_item:
                return "ground_truth"

    return "desconhecido"


def process_upload(uploaded):
    uploaded_files = []

    for name, content in uploaded.items():
        if not name.lower().endswith(".json"):
            print(f"⚠ Ignored (not JSON): {name}")
            continue

        data = load_json_bytes(name, content)
        file_type = classify_json(name, data)
        uploaded_files.append({"nome": name, "tipo": file_type, "dados": data})

    return uploaded_files


try:
    from google.colab import files
except ImportError as e:
    raise RuntimeError(
        "This notebook was prepared for interactive upload in Google Colab. "
        "Run it in Colab or adapt this cell for local file reading."
    ) from e


# ------------------------------------------------------------
# STEP 1 — Ground truth
# ------------------------------------------------------------
print("STEP 1/2 — Upload the ground truth JSON file:")
upload_gt = files.upload()
json_files = process_upload(upload_gt)

ground_truths = [a for a in json_files if a["tipo"] == "ground_truth"]
generation_files = [a for a in json_files if a["tipo"] == "geracoes"]
unknown_files = [a["nome"] for a in json_files if a["tipo"] == "desconhecido"]

if unknown_files:
    print("⚠ JSON file(s) with unrecognized structure:", unknown_files)

if len(ground_truths) != 1:
    raise ValueError(
        f"Exactly 1 ground truth file is required. Identified: {len(ground_truths)}. "
        "Check whether the file contains 'is_reference_base': true or cases with "
        "'reference_id' e 'gherkin'."
    )

ground_truth_name = ground_truths[0]["nome"]
ground_truth = ground_truths[0]["dados"]

print(f"\n✓ Ground truth identified: {ground_truth_name}")


# ------------------------------------------------------------
# STEP 2 — Generations
# ------------------------------------------------------------
if not generation_files:
    print("\nSTEP 2/2 — Now upload one or more generation JSON files:")
    upload_gen = files.upload()
    new_files = process_upload(upload_gen)

    new_ground_truths = [a for a in new_files if a["tipo"] == "ground_truth"]
    if new_ground_truths:
        print(
            "⚠ Additional ground truth ignored during the generation step:",
            [a["nome"] for a in new_ground_truths]
        )

    new_unknown_files = [
        a["nome"] for a in new_files if a["tipo"] == "desconhecido"
    ]
    if new_unknown_files:
        print("⚠ JSON file(s) with unrecognized structure:", new_unknown_files)

    generation_files.extend(
        a for a in new_files if a["tipo"] == "geracoes"
    )

if not generation_files:
    raise ValueError(
        "No generation file was identified. Generation files "
        "must contain 'cases' and, within each case, the key 'generations'."
    )

print(f"\n✓ Generation files identified: {len(generation_files)}")
for file_item in generation_files:
    data = file_item["dados"]
    print(
        f"  - {file_item['nome']} | model={data.get('model')} | "
        f"technique={data.get('technique')} | "
        f"declared executions={data.get('number_of_executions')}"
    )

In [ ]:
# ============================================================
# 4. DATA VALIDATION
# ============================================================

def index_ground_truth(ground_truth_data):
    refs = {}
    duplicates = []

    for case in ground_truth_data.get("cases", []):
        case_id = case.get("case_id")

        if not case_id:
            continue

        if case_id in refs:
            duplicates.append(case_id)

        refs[case_id] = case

    if duplicates:
        raise ValueError(
            f"Duplicate case_id value(s) in the ground truth: {sorted(set(duplicates))}"
        )

    return refs


references = index_ground_truth(ground_truth)
validation_warnings = []

for file_item in generation_files:
    name = file_item["nome"]
    data = file_item["dados"]
    file_ids = []
    declared_executions = data.get("number_of_executions")

    for case in data.get("cases", []):
        case_id = case.get("case_id")
        file_ids.append(case_id)

        if case_id not in references:
            validation_warnings.append(
                f"{name}: {case_id} exists in the generations but not in the ground truth."
            )
            continue

        reference_original = references[case_id].get("original_case")
        generated_original = case.get("original_case")

        if (
            reference_original is not None
            and generated_original is not None
            and reference_original != generated_original
        ):
            validation_warnings.append(
                f"{name}: divergent original_case in {case_id}."
            )

        generations = case.get("generations", [])

        if declared_executions is not None and len(generations) != declared_executions:
            validation_warnings.append(
                f"{name}: {case_id} contains {len(generations)} generations, "
                f"but the file declares {declared_executions}."
            )

        executions = [g.get("execution") for g in generations]
        valid_executions = [e for e in executions if e is not None]

        if len(valid_executions) != len(set(valid_executions)):
            validation_warnings.append(
                f"{name}: there are duplicate execution numbers in {case_id}."
            )

    reference_ids = set(references)
    generation_ids = set(file_ids)

    missing_ids = sorted(reference_ids - generation_ids)
    if missing_ids:
        validation_warnings.append(
            f"{name}: {len(missing_ids)} ground-truth case_id value(s) do not appear in the generations. "
            f"Examples: {missing_ids[:10]}"
        )

print(f"Cases in ground truth: {len(references)}")

if validation_warnings:
    print(f"\n⚠ Found {len(validation_warnings)} validation warning(s):")
    for warning in validation_warnings:
        print(" -", warning)
else:
    print("\n✓ Structure validated without warnings.")

In [ ]:
# ============================================================
# 5. NLI MODEL LOADING
# ============================================================

print("Loading tokenizer and NLI model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

def normalize_label_name(label_name):
    return str(label_name).strip().lower().replace("-", "_").replace(" ", "_")

def get_nli_indices(config):
    id2label = getattr(config, "id2label", {}) or {}
    label2id = getattr(config, "label2id", {}) or {}
    label_map = {}

    for idx, name in id2label.items():
        normalized_name = normalize_label_name(name)
        if "entail" in normalized_name:
            label_map["entailment"] = int(idx)
        elif "neutral" in normalized_name:
            label_map["neutral"] = int(idx)
        elif "contrad" in normalized_name:
            label_map["contradiction"] = int(idx)

    for name, idx in label2id.items():
        normalized_name = normalize_label_name(name)
        if "entail" in normalized_name:
            label_map["entailment"] = int(idx)
        elif "neutral" in normalized_name:
            label_map["neutral"] = int(idx)
        elif "contrad" in normalized_name:
            label_map["contradiction"] = int(idx)

    missing = {"entailment", "neutral", "contradiction"} - set(label_map)
    if missing:
        raise ValueError(
            "Could not automatically identify all NLI labels. "
            f"Missing: {sorted(missing)} | "
            f"id2label={id2label} | label2id={label2id}"
        )

    return label_map

NLI_INDICES = get_nli_indices(model.config)

print("✓ Model loaded.")
print("NLI mapping:", NLI_INDICES)

In [ ]:
# ============================================================
# 6. FUNCTIONS: NORMALIZATION, GHERKIN, ALIGNMENT, AND NLI
# ============================================================

STEP_PATTERN = re.compile(
    r"^\s*(Given|When|Then|And|But|Dado|Dada|Dados|Dadas|Quando|Então|Entao|E|Mas)\b\s*(.*)$",
    flags=re.IGNORECASE
)

GROUP_MAP = {
    "given": "Given",
    "dado": "Given",
    "dada": "Given",
    "dados": "Given",
    "dadas": "Given",
    "when": "When",
    "quando": "When",
    "then": "Then",
    "então": "Then",
    "entao": "Then",
}

CONTINUATIONS = {"and", "but", "e", "mas"}


def normalize_gherkin(value):
    """
    Converts different Gherkin representations into multiline text.

    Supports:
    - regular string with line breaks;
    - string containing literal \\n;
    - lists;
    - JSON dictionaries/objects;
    - Markdown blocks ```gherkin ... ```.
    """
    if value is None:
        return ""

    if isinstance(value, str):
        text = value

        # Converts literal "\n" sequences into real line breaks,
        # only when there are no relevant real line breaks.
        if "\\n" in text:
            text = text.replace("\\r\\n", "\n").replace("\\n", "\n")

        # Removes Markdown fences without removing the content.
        text = re.sub(
            r"^\s*```(?:gherkin|cucumber|feature)?\s*$",
            "",
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )
        text = re.sub(
            r"^\s*```\s*$",
            "",
            text,
            flags=re.MULTILINE
        )

        return text.strip()

    if isinstance(value, list):
        parts = [normalize_gherkin(item) for item in value]
        return "\n".join(p for p in parts if p).strip()

    if isinstance(value, dict):
        # Prioritizes keys that normally contain Gherkin text.
        priority_keys = [
            "gherkin",
            "text",
            "texto",
            "content",
            "conteudo",
            "scenario",
            "cenario",
            "steps",
            "passos",
            "given",
            "when",
            "then",
            "and",
        ]

        parts = []

        for key_item in priority_keys:
            if key_item in value:
                normalized = normalize_gherkin(value[key_item])
                if normalized:
                    parts.append(normalized)

        if parts:
            return "\n".join(parts).strip()

        # Fallback: iterates over all values.
        for item in value.values():
            normalized = normalize_gherkin(item)
            if normalized:
                parts.append(normalized)

        return "\n".join(parts).strip()

    return str(value).strip()


def extract_gherkin_steps(value):
    """
    Extracts Gherkin steps from different formats.

    And/But/E/Mas inherit the last main context:
    Given, When ou Then.
    """
    text = normalize_gherkin(value)

    steps = []
    current_group = None
    group_positions = defaultdict(int)

    for line in text.splitlines():
        line = line.strip()

        # Removes common markers without affecting the keyword.
        line = re.sub(r"^\s*[-*•]\s*", "", line)

        m = STEP_PATTERN.match(line)
        if not m:
            continue

        original_keyword = m.group(1).strip()
        content = m.group(2).strip()
        normalized_keyword = original_keyword.lower()

        if normalized_keyword in GROUP_MAP:
            current_group = GROUP_MAP[normalized_keyword]
        elif normalized_keyword in CONTINUATIONS:
            current_group = current_group or "SemContexto"
        else:
            current_group = current_group or "SemContexto"

        group_positions[current_group] += 1

        steps.append({
            "ordem_global": len(steps) + 1,
            "keyword": original_keyword,
            "grupo": current_group,
            "posicao_no_grupo": group_positions[current_group],
            "texto": content,
        })

    return steps


def align_steps(reference_steps, generated_steps):
    reference_by_key = {
        (p["grupo"], p["posicao_no_grupo"]): p
        for p in reference_steps
    }
    generated_by_key = {
        (p["grupo"], p["posicao_no_grupo"]): p
        for p in generated_steps
    }

    group_order = {
        "Given": 0,
        "When": 1,
        "Then": 2,
        "SemContexto": 3
    }

    keys = sorted(
        set(reference_by_key) | set(generated_by_key),
        key=lambda x: (group_order.get(x[0], 4), x[1])
    )

    pairs = []
    missing = 0
    extras = 0

    for key_item in keys:
        ref = reference_by_key.get(key_item)
        gen = generated_by_key.get(key_item)

        if ref is not None and gen is not None:
            pairs.append({
                "grupo": key_item[0],
                "posicao_no_grupo": key_item[1],
                "passo_ref": ref,
                "passo_gen": gen,
            })
        elif ref is not None:
            missing += 1
        elif gen is not None:
            extras += 1

    return pairs, missing, extras


@torch.inference_mode()
def infer_nli_pairs(text_pairs, batch_size=None, description="NLI"):
    if not text_pairs:
        return []

    batch_size = batch_size or BATCH_SIZE
    outputs = []

    total_batches = (len(text_pairs) + batch_size - 1) // batch_size

    for start in tqdm(
        range(0, len(text_pairs), batch_size),
        total=total_batches,
        desc=description
    ):
        batch = text_pairs[start:start + batch_size]
        premises = [p for p, _ in batch]
        hypotheses = [h for _, h in batch]

        inputs = tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in inputs.items()
        }

        if DEVICE.type == "cuda" and USE_MIXED_PRECISION:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(**inputs).logits
        else:
            logits = model(**inputs).logits

        probs = torch.softmax(
            logits.float(),
            dim=-1
        ).cpu().numpy()

        for p in probs:
            pe = float(p[NLI_INDICES["entailment"]])
            pn = float(p[NLI_INDICES["neutral"]])
            pc = float(p[NLI_INDICES["contradiction"]])

            score = pe + 0.5 * pn

            probabilities = {
                "entailment": pe,
                "neutral": pn,
                "contradiction": pc,
            }

            outputs.append({
                "entailment": pe,
                "neutral": pn,
                "contradiction": pc,
                "nli_score": score,
                "classe_nli": max(
                    probabilities,
                    key=probabilities.get
                ),
            })

    return outputs


def combine_bidirectional_nli(
    nli_ref_gen,
    nli_gen_ref
):
    pe = (
        nli_ref_gen["entailment"]
        + nli_gen_ref["entailment"]
    ) / 2

    pn = (
        nli_ref_gen["neutral"]
        + nli_gen_ref["neutral"]
    ) / 2

    pc = (
        nli_ref_gen["contradiction"]
        + nli_gen_ref["contradiction"]
    ) / 2

    score = (
        nli_ref_gen["nli_score"]
        + nli_gen_ref["nli_score"]
    ) / 2

    probabilities = {
        "entailment": pe,
        "neutral": pn,
        "contradiction": pc,
    }

    return {
        "entailment": pe,
        "neutral": pn,
        "contradiction": pc,
        "nli_score": score,
        "classe_nli": max(
            probabilities,
            key=probabilities.get
        ),
        "nli_ref_para_gerado": nli_ref_gen["nli_score"],
        "nli_gerado_para_ref": nli_gen_ref["nli_score"],
    }


# Mandatory internal regex test using Portuguese aliases for backward compatibility.
_test_text = """Scenario: teste
Given uma condição
And outra condição
When uma ação
Then um resultado"""

_test_steps = extract_gherkin_steps(_test_text)

assert len(_test_steps) == 4, (
    "Internal failure in the Gherkin extractor. "
    f"Found {len(_test_steps)} steps in the test."
)

print("✓ Gherkin extractor validated internally.")

In [ ]:
# ============================================================
# 7. STEP-LEVEL COMPARISON PREPARATION
# ============================================================

base_generations = []
base_step_pairs = []

# Shows a real sample before processing everything.
first_reference = next(iter(references.values()))

print("GROUND TRUTH STRUCTURE SAMPLE")
print("Type of gherkin field:", type(first_reference.get("gherkin")).__name__)
print(
    normalize_gherkin(first_reference.get("gherkin"))[:1000]
)
print("-" * 80)

first_generation_shown = False

for file_item in generation_files:
    generations_name = file_item["nome"]
    data = file_item["dados"]
    model_name = data.get("model", "")
    technique = data.get("technique", "")
    declared_executions_value = data.get(
        "number_of_executions"
    )

    for generated_case in data.get("cases", []):
        case_id = generated_case.get("case_id")
        reference = references.get(case_id)

        if reference is None:
            continue

        reference_value = reference.get("gherkin")
        reference_gherkin = normalize_gherkin(reference_value)
        reference_steps = extract_gherkin_steps(reference_value)

        for geracao in generated_case.get(
            "generations",
            []
        ):
            generated_value = geracao.get("gherkin")
            generated_gherkin = normalize_gherkin(generated_value)
            generated_steps = extract_gherkin_steps(generated_value)

            if not first_generation_shown:
                print(
                    "GENERATION STRUCTURE SAMPLE"
                )
                print(
                    "Type of gherkin field:",
                    type(generated_value).__name__
                )
                print(generated_gherkin[:1000])
                print("-" * 80)
                first_generation_shown = True

            pairs, missing_steps, extra_steps = (
                align_steps(
                    reference_steps,
                    generated_steps
                )
            )

            internal_id = len(base_generations)

            coverage = (
                len(pairs) / len(reference_steps)
                if len(reference_steps) > 0
                else np.nan
            )

            base_generations.append({
                "_id_interno": internal_id,
                "arquivo_ground_truth": ground_truth_name,
                "arquivo_geracoes": generations_name,
                "modelo": model_name,
                "tecnica": technique,
                "execucoes_declaradas": declared_executions_value,
                "case_id": case_id,
                "source_id": reference.get("source_id"),
                "source_line": reference.get("source_line"),
                "original_case": reference.get(
                    "original_case"
                ),
                "reference_id": reference.get(
                    "reference_id"
                ),
                "generation_id": geracao.get(
                    "generation_id"
                ),
                "execucao": geracao.get("execution"),
                "gherkin_ground_truth": reference_gherkin,
                "gherkin_gerado": generated_gherkin,
                "passos_ground_truth": len(reference_steps),
                "passos_gerados": len(generated_steps),
                "passos_alinhados": len(pairs),
                "passos_ausentes": missing_steps,
                "passos_extras": extra_steps,
                "coverage": coverage,
            })

            for pair in pairs:
                base_step_pairs.append({
                    "_id_interno": internal_id,
                    "arquivo_geracoes": generations_name,
                    "modelo": model_name,
                    "tecnica": technique,
                    "case_id": case_id,
                    "generation_id": geracao.get(
                        "generation_id"
                    ),
                    "execucao": geracao.get(
                        "execution"
                    ),
                    "grupo": pair["grupo"],
                    "posicao_no_grupo": pair[
                        "posicao_no_grupo"
                    ],
                    "keyword_ground_truth": pair[
                        "passo_ref"
                    ]["keyword"],
                    "keyword_gerado": pair[
                        "passo_gen"
                    ]["keyword"],
                    "passo_ground_truth": pair[
                        "passo_ref"
                    ]["texto"],
                    "passo_gerado": pair[
                        "passo_gen"
                    ]["texto"],
                })

if not base_generations:
    raise ValueError(
        "No generation could be prepared."
    )

print(
    f"✓ Generations prepared: "
    f"{len(base_generations):,}"
)
print(
    f"✓ Aligned step pairs: "
    f"{len(base_step_pairs):,}"
)

total_ground_truth_steps = sum(
    x["passos_ground_truth"]
    for x in base_generations
)
total_generated_steps = sum(
    x["passos_gerados"]
    for x in base_generations
)
total_aligned_steps = sum(
    x["passos_alinhados"]
    for x in base_generations
)

print()
print("GHERKIN EXTRACTION VALIDATION")
print(
    f"Total ground-truth steps: "
    f"{total_ground_truth_steps:,}"
)
print(
    f"Total generated steps: "
    f"{total_generated_steps:,}"
)
print(
    f"Total aligned steps: "
    f"{total_aligned_steps:,}"
)

if total_ground_truth_steps == 0:
    raise RuntimeError(
        "The ground-truth gherkin field exists, "
        "but no Given/When/Then/And step was recognized. "
        "See the SAMPLE printed above."
    )

if total_generated_steps == 0:
    raise RuntimeError(
        "The generations' gherkin field exists, "
        "but no Given/When/Then/And step was recognized. "
        "See the SAMPLE printed above."
    )

if total_aligned_steps == 0:
    raise RuntimeError(
        "The steps were recognized, but none "
        "could be aligned between reference and generation."
    )

print(
    "✓ Extraction and alignment validated. "
    "You can run cell 8."
)

In [ ]:
# ============================================================
# 8. OPTIMIZED BIDIRECTIONAL STEP-LEVEL NLI
# ============================================================

if DEVICE.type == "cpu":
    print("⚠ You are running on CPU.")
    print("  For this dataset, using a GPU is strongly recommended.")
    print()

directional_pairs = []

for item in base_step_pairs:
    ref = item["passo_ground_truth"]
    gen = item["passo_gerado"]

    directional_pairs.append((ref, gen))
    directional_pairs.append((gen, ref))

total_without_cache = len(directional_pairs)

# Removes only EXACTLY repeated pairs.
# The mathematical result is unchanged.
unique_pairs = list(dict.fromkeys(directional_pairs))

saved_inferences = total_without_cache - len(unique_pairs)
savings_percentage = (
    100 * saved_inferences / total_without_cache
    if total_without_cache > 0
    else 0
)

print(f"Aligned pairs: {len(base_step_pairs):,}")
print(f"Bidirectional inferences without cache: {total_without_cache:,}")
print(f"Unique directional pairs: {len(unique_pairs):,}")
print(
    f"Cache eliminated {saved_inferences:,} repeated inferences "
    f"({savings_percentage:.1f}%)."
)
print(f"Device used: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print()

calculation_start = time.time()

unique_results = infer_nli_pairs(
    unique_pairs,
    batch_size=BATCH_SIZE,
    description="Bidirectional NLI"
)

total_time = time.time() - calculation_start

nli_cache = {
    pair: result
    for pair, result in zip(unique_pairs, unique_results)
}

print()
print(f"✓ Inference completed in {total_time / 60:.2f} minutes.")

step_details = []
by_generation = defaultdict(list)

for base in base_step_pairs:
    ref = base["passo_ground_truth"]
    gen = base["passo_gerado"]

    rg = nli_cache[(ref, gen)]
    gr = nli_cache[(gen, ref)]

    bidirectional_nli = combine_bidirectional_nli(rg, gr)

    record = {
        **base,
        **bidirectional_nli,
    }

    step_details.append(record)
    by_generation[base["_id_interno"]].append(bidirectional_nli)

results = []

for base in base_generations:
    step_scores = by_generation.get(base["_id_interno"], [])

    if step_scores:
        nli_score = float(np.mean([
            x["nli_score"]
            for x in step_scores
        ]))

        entailment_mean = float(np.mean([
            x["entailment"]
            for x in step_scores
        ]))

        neutral_mean = float(np.mean([
            x["neutral"]
            for x in step_scores
        ]))

        contradiction_mean = float(np.mean([
            x["contradiction"]
            for x in step_scores
        ]))

        contradiction_rate = float(np.mean([
            x["classe_nli"] == "contradiction"
            for x in step_scores
        ]))
    else:
        nli_score = np.nan
        entailment_mean = np.nan
        neutral_mean = np.nan
        contradiction_mean = np.nan
        contradiction_rate = np.nan

    def round_value(v):
        return (
            round(v, NLI_DECIMAL_PLACES)
            if not np.isnan(v)
            else np.nan
        )

    results.append({
        **{
            k: v
            for k, v in base.items()
            if k != "_id_interno"
        },
        "nli_score": round_value(nli_score),
        "entailment_mean": round_value(entailment_mean),
        "neutral_mean": round_value(neutral_mean),
        "contradiction_mean": round_value(contradiction_mean),
        "contradiction_rate": round_value(contradiction_rate),
    })

results_df = pd.DataFrame(results)
step_details_df = pd.DataFrame(step_details)

if not step_details_df.empty:
    for column in [
        "entailment",
        "neutral",
        "contradiction",
        "nli_score",
        "nli_ref_para_gerado",
        "nli_gerado_para_ref",
    ]:
        step_details_df[column] = (
            step_details_df[column]
            .round(NLI_DECIMAL_PLACES)
        )

ranking_keys = [
    "arquivo_geracoes",
    "modelo",
    "tecnica",
    "case_id",
]

results_df = results_df.sort_values(
    ranking_keys + ["nli_score", "execucao"],
    ascending=[True, True, True, True, False, True],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

results_df["ranking_no_caso"] = (
    results_df
    .groupby(ranking_keys, dropna=False)
    .cumcount()
    + 1
)

columns = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "nli_score",
    "ranking_no_caso",
    "entailment_mean",
    "neutral_mean",
    "contradiction_mean",
    "contradiction_rate",
    "coverage",
    "passos_ground_truth",
    "passos_gerados",
    "passos_alinhados",
    "passos_ausentes",
    "passos_extras",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

results_df = results_df[columns]

print()
print(f"✓ Comparisons calculated: {len(results_df):,}")
print(f"✓ Cases evaluated: {results_df['case_id'].nunique():,}")
print(
    "✓ Scenarios without aligned steps: "
    f"{results_df['nli_score'].isna().sum():,}"
)

In [ ]:
# ============================================================
# 9. ANALYSIS TABLES
# ============================================================

overall_summary_df = (
    results_df
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("nli_score", "count"),
        nli_media=("nli_score", "mean"),
        nli_mediana=("nli_score", "median"),
        desvio_padrao=("nli_score", "std"),
        nli_minimo=("nli_score", "min"),
        nli_maximo=("nli_score", "max"),
        entailment_medio=("entailment_mean", "mean"),
        neutral_medio=("neutral_mean", "mean"),
        contradiction_medio=("contradiction_mean", "mean"),
        taxa_contradicao_media=("contradiction_rate", "mean"),
        coverage_medio=("coverage", "mean"),
    )
    .reset_index()
)

print("OVERALL SUMMARY")
display(overall_summary_df.style.format({
    "nli_media": "{:.4f}", "nli_mediana": "{:.4f}", "desvio_padrao": "{:.4f}",
    "nli_minimo": "{:.4f}", "nli_maximo": "{:.4f}", "entailment_medio": "{:.4f}",
    "neutral_medio": "{:.4f}", "contradiction_medio": "{:.4f}",
    "taxa_contradicao_media": "{:.2%}", "coverage_medio": "{:.2%}",
}))


case_summary_df = (
    results_df
    .groupby(["arquivo_geracoes", "modelo", "tecnica", "case_id", "original_case"], dropna=False)
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        nli_media=("nli_score", "mean"),
        nli_mediana=("nli_score", "median"),
        desvio_padrao=("nli_score", "std"),
        melhor_nli=("nli_score", "max"),
        pior_nli=("nli_score", "min"),
        contradiction_media=("contradiction_mean", "mean"),
        contradiction_rate_media=("contradiction_rate", "mean"),
        coverage_media=("coverage", "mean"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

case_summary_df["cv_nli_percentual"] = np.where(
    case_summary_df["nli_media"] != 0,
    (case_summary_df["desvio_padrao"] / case_summary_df["nli_media"]) * 100,
    np.nan
)

print("\nSUMMARY BY CASE — first 30 rows")
display(case_summary_df.head(30).style.format({
    "nli_media": "{:.4f}", "nli_mediana": "{:.4f}", "desvio_padrao": "{:.4f}",
    "melhor_nli": "{:.4f}", "pior_nli": "{:.4f}", "contradiction_media": "{:.4f}",
    "contradiction_rate_media": "{:.2%}", "coverage_media": "{:.2%}",
    "cv_nli_percentual": "{:.2f}%",
}))

print("\nCOMPARISONS — first 50 rows")
display_columns = [
    "modelo", "tecnica", "case_id", "original_case", "execucao", "nli_score",
    "ranking_no_caso", "entailment_mean", "neutral_mean", "contradiction_mean",
    "contradiction_rate", "coverage", "passos_ground_truth", "passos_gerados",
    "passos_alinhados", "passos_ausentes", "passos_extras",
]
display(results_df[display_columns].head(50).style.format({
    "nli_score": "{:.4f}", "entailment_mean": "{:.4f}", "neutral_mean": "{:.4f}",
    "contradiction_mean": "{:.4f}", "contradiction_rate": "{:.2%}", "coverage": "{:.2%}",
}))


In [ ]:
# ============================================================
# 10. QUICK CASE LOOKUP
# ============================================================

def view_case(case_id, show_steps=True):
    case_slice = results_df[results_df["case_id"] == case_id].copy()
    if case_slice.empty:
        print(f"No result found for {case_id}.")
        return

    columns = [
        "modelo", "tecnica", "case_id", "original_case", "execucao", "nli_score",
        "ranking_no_caso", "entailment_mean", "neutral_mean", "contradiction_mean",
        "contradiction_rate", "coverage", "passos_ground_truth", "passos_gerados",
        "passos_alinhados", "passos_ausentes", "passos_extras",
        "gherkin_ground_truth", "gherkin_gerado",
    ]

    display(case_slice[columns].sort_values(
        ["modelo", "tecnica", "nli_score", "execucao"],
        ascending=[True, True, False, True]
    ).style.format({
        "nli_score": "{:.4f}", "entailment_mean": "{:.4f}", "neutral_mean": "{:.4f}",
        "contradiction_mean": "{:.4f}", "contradiction_rate": "{:.2%}", "coverage": "{:.2%}",
    }))

    if show_steps and not step_details_df.empty:
        details = step_details_df[step_details_df["case_id"] == case_id].copy()
        if not details.empty:
            print("\nBIDIRECTIONAL NLI BY STEP")
            step_columns = [
                "modelo", "tecnica", "case_id", "execucao", "grupo", "posicao_no_grupo",
                "passo_ground_truth", "passo_gerado", "nli_ref_para_gerado",
                "nli_gerado_para_ref", "nli_score", "entailment", "neutral",
                "contradiction", "classe_nli",
            ]
            display(details[step_columns].sort_values(
                ["modelo", "tecnica", "execucao", "grupo", "posicao_no_grupo"]
            ).style.format({
                "nli_ref_para_gerado": "{:.4f}", "nli_gerado_para_ref": "{:.4f}",
                "nli_score": "{:.4f}", "entailment": "{:.4f}", "neutral": "{:.4f}",
                "contradiction": "{:.4f}",
            }))

first_case_id = results_df["case_id"].iloc[0]
print(f"Query example: {first_case_id}")
view_case(first_case_id)

# visualizar_caso("TC_261")

In [ ]:
# ============================================================
# 11. EXPORT — A SINGLE CSV
# ============================================================

def slug(text):
    text = str(text or "").strip().lower()
    text = re.sub(r"[^a-z0-9._-]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-")
    return text or "sem-identificacao"

unique_metadata = results_df[["modelo", "tecnica"]].drop_duplicates().reset_index(drop=True)

if len(unique_metadata) == 1:
    model_name = unique_metadata.loc[0, "modelo"]
    technique_name = unique_metadata.loc[0, "tecnica"]
    csv_file_name = f"metricas_nli_passos_bidirecional_{slug(model_name)}_{slug(technique_name)}.csv"
else:
    csv_file_name = "metricas_nli_passos_bidirecional_multiplos_modelos_tecnicas.csv"

results_df.to_csv(csv_file_name, index=False, encoding="utf-8-sig")

print(f"✓ CSV generated: {csv_file_name}")
print(f"✓ Rows exported: {len(results_df):,}")
print("✓ Only the CSV aggregated by scenario/execution is exported.")

if AUTO_DOWNLOAD_CSV:
    try:
        from google.colab import files
        files.download(csv_file_name)
    except Exception as e:
        print(f"Automatic download was not completed: {e}")